In [1]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx  # chỉ số cột
        self.threshold = threshold      # ngưỡng chia
        self.left = left                # nhánh trái
        self.right = right              # nhánh phải
        self.value = value              # giá trị nếu là node lá

In [2]:
import numpy as np

class DecisionTreeRegressor:
    def __init__(self, max_depth=5, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None

In [3]:
    def _calculate_variance_reduction(self, y, y_left, y_right):
        var_total = np.var(y)
        weight_left = len(y_left) / len(y)
        weight_right = len(y_right) / len(y)

        reduction = var_total - (weight_left * np.var(y_left) + weight_right * np.var(y_right))
        return reduction

In [4]:
    def _get_best_split(self, X, y):
        best_split = {}
        best_var_reduction = -float("inf")

        n_samples, n_features = X.shape

        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])

            for threshold in thresholds:
                left_idx = X[:, feature_idx] <= threshold
                right_idx = X[:, feature_idx] > threshold

                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue

                var_reduction = self._calculate_variance_reduction(
                    y, y[left_idx], y[right_idx]
                )

                if var_reduction > best_var_reduction:
                    best_var_reduction = var_reduction
                    best_split = {
                        "feature_idx": feature_idx,
                        "threshold": threshold,
                        "left_idx": left_idx,
                        "right_idx": right_idx
                    }

        return best_split

In [5]:
    def _build_tree(self, X, y, depth=0):
        n_samples = X.shape[0]

        # điều kiện dừng
        if (n_samples < self.min_samples_split) or (depth >= self.max_depth):
            leaf_value = np.mean(y)
            return Node(value=leaf_value)

        split = self._get_best_split(X, y)

        if not split:
            return Node(value=np.mean(y))

        left = self._build_tree(X[split["left_idx"]], y[split["left_idx"]], depth + 1)
        right = self._build_tree(X[split["right_idx"]], y[split["right_idx"]], depth + 1)

        return Node(
            feature_idx=split["feature_idx"],
            threshold=split["threshold"],
            left=left,
            right=right
        )

In [6]:
    def fit(self, X, y):
        self.root = self._build_tree(X, y)

In [7]:
    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

In [8]:
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])